# 28 — Triplet Autoencoder WJ 512 (fixed)

Same architecture as nb27 (autoencoder) but trained with **in-batch hard-negative WJ triplet loss + FN filter + reconstruction**,
matching nb15's training protocol exactly. Fixes original: random negatives → hard negatives, added FN masking, batch=2048, 50 epochs.

In [14]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    build_fn_mask, build_gt_cache, build_gt_gpu,
    cleanup, eval_recall, l1_simplex, load_dataset,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name = "10k"
out_dim      = 512
device       = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS      = 150
seed         = 42
batch_size   = 2048
epochs       = 75
lr           = 1e-3
weight_decay = 1e-4
max_pos      = 30
margin       = 0.3
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]

METHOD_NAME   = "triplet_autoencoder_wj_512"
NOTEBOOK_NAME = "28_triplet_autoencoder_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_autoencoder_wj_512.pkl"
CKPT_PATH     = "/tmp/best_sota_triplet_autoencoder_wj_512_full.pt"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"device={device} | batch={batch_size} | epochs={epochs}")

device=cuda:0 | batch=2048 | epochs=75


In [15]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [16]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_triplet_loss_inbatch(anchors, positives, margin=0.3, gt_matrix=None):
    """In-batch hard-negative WJ triplet loss with FN masking."""
    sim_ap = wj_sim(anchors, positives)
    mins_c = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    maxs_c = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    sim_cross = mins_c / maxs_c.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    n_fn = 0
    if gt_matrix is not None:
        fn_mask = gt_matrix.to(sim_cross.device)
        n_fn = int(fn_mask.sum().item())
        if n_fn:
            sim_cross[fn_mask] = -1e9
    sim_an   = sim_cross.max(dim=1).values
    loss     = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0, n_fn
    return loss[violated].mean(), int(violated.sum().item()), n_fn

class IndexAnchorPositiveDataset(Dataset):
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}  steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return qid, pid

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        return self.encode(x)

def embed_all(model, qt, batch_size=512):
    enc = model.module if hasattr(model, "module") else model
    enc.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(enc.encode(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS={rr_metrics['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [17]:
# All 8 GPUs; vecs on cuda:7 so cuda:0 (gather) stays free for the 8 GB triplet intermediate.
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

dataset = IndexAnchorPositiveDataset(gt, query_start, max_pos=max_pos)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                     num_workers=4, pin_memory=True, drop_last=True,
                     persistent_workers=True)

gt_stacked = build_gt_cache(gt, len(qt_norm), query_start, dataset_name)
gt_gpu     = build_gt_gpu(gt_stacked, vecs_device)
del gt_stacked

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best = float('inf')
t0_train = time.time()

epoch_bar = tqdm(range(1, epochs + 1), desc="epochs", unit="ep")
for epoch in epoch_bar:
    model.train()
    tot_loss = tot_trip = tot_viol = steps = 0
    step_bar = tqdm(loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
    for a_ids, p_ids in step_bar:
        a = vecs_gpu[a_ids.to(vecs_device)].to(device)
        p = vecs_gpu[p_ids.to(vecs_device)].to(device)
        B = a.shape[0]
        z = model(torch.cat([a, p]))
        za, zp = z[:B], z[B:]
        fn_mask = build_fn_mask(a_ids, p_ids, gt_gpu, query_start)
        trip, n_viol, _ = wj_triplet_loss_inbatch(za, zp, margin=margin, gt_matrix=fn_mask)
        loss = trip
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_trip += float(trip.detach())
        tot_viol += n_viol; steps += 1
        step_bar.set_postfix(loss=f"{float(loss.detach()):.4f}", viol=n_viol)
    sch.step()
    avg = tot_loss / max(steps, 1)
    if avg < best:
        best = avg
        torch.save(model.module.state_dict(), CKPT_PATH)
    elapsed = (time.time() - t0_train) / 60
    eta     = elapsed / epoch * (epochs - epoch)
    epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best:.4f}", eta=f"{eta:.0f}m")
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | trip={tot_trip/steps:.4f} | "
              f"viol={tot_viol/steps:.1f} | {elapsed:.1f}min | eta={eta:.1f}min", flush=True)

print(f"Training done. best={best:.4f} | saved {CKPT_PATH}")

Pre-loading vectors to cuda:7...
Loaded: 0.69 GB on cuda:7
DataParallel on 8 GPUs
pairs=46,722  steps/epoch=22
gt_stacked loaded from cache (2000, 678) in 0.0s
gt_gpu on cuda:7: (2000, 678) (0.01 GB) in 0.0s


epochs:   0%|          | 0/75 [00:00<?, ?ep/s]

ep01:   0%|          | 0/22 [00:00<?, ?step/s]

epoch 01/75 | loss=0.1814 | trip=0.1814 | viol=1865.1 | 0.1min | eta=9.0min


ep02:   0%|          | 0/22 [00:00<?, ?step/s]

...

ep72:   0%|          | 0/22 [00:00<?, ?step/s]

ep73:   0%|          | 0/22 [00:00<?, ?step/s]

ep74:   0%|          | 0/22 [00:00<?, ?step/s]

ep75:   0%|          | 0/22 [00:00<?, ?step/s]

epoch 75/75 | loss=0.0876 | trip=0.0876 | viol=1092.1 | 8.9min | eta=0.0min
Training done. best=0.0876 | saved /tmp/best_sota_triplet_autoencoder_wj_512_full.pt


In [18]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.7021
R@50   = 0.8544
R@100  = 0.8850
R@500  = 0.9757
QPS=8010.8
saved triplet_autoencoder_wj_512 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl
Corpus pre-loaded to GPU: 0.55 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_autoencoder_wj_512_rerank_500 R@10 = 0.9965
triplet_autoencoder_wj_512_rerank_500 R@50 = 0.9984
triplet_autoencoder_wj_512_rerank_500 R@100 = 0.9987
triplet_autoencoder_wj_512_rerank_500 R@500 = 0.9758
triplet_autoencoder_wj_512_rerank_500 QPS=3235.7
saved triplet_autoencoder_wj_512_rerank_500 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_autoencoder_wj_512_rerank_1000 R@10 = 0.9965
triplet_autoencoder_wj_512_rerank_1000 R@50 = 0.9985
triplet_autoencoder_wj_512_rerank_1000 R@100 = 0.9988
triplet_autoencoder_wj_512_rerank_1000 R@500 = 0.9876
triplet_autoencoder_wj_512_rerank_1000 QPS=1889.9
saved triplet_autoencoder_wj_512_rerank_1000 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl


In [19]:
# ── 10k dataset eval (same model, different data) ───────────────────────────
import pickle
from sota_experiment_common import (
    load_dataset, l1_simplex, nmslib_neighbors, eval_recall,
    preload_rerank_corpus, rerank_wj_gpu, release_rerank_corpus, save_result
)

OUT_PATH_10K = "/tmp/results_sota_triplet_autoencoder_wj_512.pkl"
METHOD_10K   = "triplet_autoencoder_wj_512"

qt_10k, gt_10k, qs_10k, corpus_qt_10k, query_qt_10k, corpus_sums_10k = load_dataset("10k")
qt_10k_norm = l1_simplex(qt_10k.copy())
cand_ks_10k = [500, 1000]

enc_10k = model.module if hasattr(model, "module") else model
enc_10k.eval()
chunks_10k = []
with torch.no_grad():
    for s in range(0, len(qt_10k_norm), 512):
        x = torch.tensor(qt_10k_norm[s:s+512], dtype=torch.float32, device=device)
        chunks_10k.append(enc_10k.encode(x).cpu().numpy().astype(np.float32))
embs_10k = np.vstack(chunks_10k)
corpus_embs_10k = embs_10k[:qs_10k]
query_embs_10k  = embs_10k[qs_10k:]
print(f"10k embs: corpus={corpus_embs_10k.shape} queries={query_embs_10k.shape}")

max_k_10k = max(max(cand_ks_10k), 500)
nbrs_10k, info_10k = nmslib_neighbors(corpus_embs_10k, query_embs_10k,
                                       space="WeightedJaccard", k=max_k_10k, threads=THREADS)
metrics_10k = {**eval_recall(gt_10k, nbrs_10k, qs_10k, max_k_10k), **info_10k, "dim": out_dim}
print("\n--- 10k no-rerank ---")
for k, v in metrics_10k.items():
    if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={metrics_10k['qps']:.1f}")
save_result(OUT_PATH_10K, "10k", METHOD_10K, metrics_10k, meta={"notebook": NOTEBOOK_NAME})

preload_rerank_corpus(corpus_qt_10k, corpus_sums_10k)
for ck in cand_ks_10k:
    cand_10k, ci_10k = nmslib_neighbors(corpus_embs_10k, query_embs_10k,
                                         space="WeightedJaccard", k=ck, threads=THREADS)
    t0 = time.time()
    rr_10k = rerank_wj_gpu(query_qt_10k, cand_10k, corpus_qt_10k, corpus_sums_10k, top_k=ck, batch_size=64)
    qps_10k = len(query_qt_10k) / max(time.time()-t0 + len(query_qt_10k)/max(ci_10k['qps'],1e-9), 1e-9)
    rr_metrics_10k = {**eval_recall(gt_10k, rr_10k, qs_10k, ck), "qps": qps_10k, "candidate_k": ck}
    key_10k = f"{METHOD_10K}_rerank_{ck}"
    print(f"\n--- 10k rerank@{ck} ---")
    for k, v in rr_metrics_10k.items():
        if isinstance(k, int): print(f"{key_10k} R@{k} = {v:.4f}")
    print(f"{key_10k} QPS={rr_metrics_10k['qps']:.1f}")
    save_result(OUT_PATH_10K, "10k", key_10k, rr_metrics_10k, meta={"notebook": NOTEBOOK_NAME})
release_rerank_corpus()


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)
10k embs: corpus=(8000, 512) queries=(2000, 512)



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*


--- 10k no-rerank ---
R@10   = 0.7021
R@50   = 0.8544
R@100  = 0.8850
R@500  = 0.9756
QPS=8766.4
saved triplet_autoencoder_wj_512 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl
Corpus pre-loaded to GPU: 0.55 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*


--- 10k rerank@500 ---
triplet_autoencoder_wj_512_rerank_500 R@10 = 0.9965
triplet_autoencoder_wj_512_rerank_500 R@50 = 0.9984
triplet_autoencoder_wj_512_rerank_500 R@100 = 0.9986
triplet_autoencoder_wj_512_rerank_500 R@500 = 0.9755
triplet_autoencoder_wj_512_rerank_500 QPS=3092.1
saved triplet_autoencoder_wj_512_rerank_500 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*


--- 10k rerank@1000 ---
triplet_autoencoder_wj_512_rerank_1000 R@10 = 0.9964
triplet_autoencoder_wj_512_rerank_1000 R@50 = 0.9984
triplet_autoencoder_wj_512_rerank_1000 R@100 = 0.9988
triplet_autoencoder_wj_512_rerank_1000 R@500 = 0.9872
triplet_autoencoder_wj_512_rerank_1000 QPS=1883.2
saved triplet_autoencoder_wj_512_rerank_1000 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl
